# 第 04 天：ICIR

> 来自《30 天因子研究计划》第 4 天  
> 主题：ICIR  
> 必做：IC 稳定性分析  
> 选做：滚动 IC  
> 目标产出：ICIR 分析图

---

## 0. 今天你要真正学会什么？

第 3 天我们学会了计算每日 IC。

但只知道平均 IC 还不够。你还要问：

> 这个因子的 IC 是稳定地为正，还是偶尔爆发、平时乱跳？

这就是 ICIR 要解决的问题。

ICIR 的英文是 `Information Coefficient Information Ratio`，可以理解为：


ICIR = IC 均值 / IC 标准差


它衡量的是：

> 单位 IC 波动里，因子贡献了多少平均 IC。

学完以后，你应该能回答：

1. 为什么平均 IC 高不一定好？
2. ICIR 是怎么计算的？
3. ICIR 和夏普比率有什么相似之处？
4. 如何分析 IC 序列的稳定性？
5. 如何画滚动 IC 和累计 IC？

一句话版：

> IC 看方向和强度，ICIR 看这个方向和强度是否稳定。

---

## 1. 先建立直觉：两个因子谁更可靠？

假设有两个因子：


因子 A：IC 经常在 0.03 附近，偶尔略低
因子 B：有时 IC = 0.20，有时 IC = -0.15


如果只看某几天，因子 B 可能很惊艳。  
但如果你要把它放进长期策略，因子 A 可能更可靠。

这就像两位投手：

- 一个球速不夸张，但每次都很稳。
- 一个偶尔投出神球，但经常失控。

因子研究不只需要“强”，还需要“稳”。

---

## 2. ICIR 的定义

给定每日 IC 序列：


IC_1, IC_2, IC_3, ..., IC_T


ICIR 定义为：


ICIR = mean(IC) / std(IC)


如果你想年化，有些研究会写成：


Annualized ICIR = mean(IC) / std(IC) * sqrt(periods_per_year)


不过初学阶段先用非年化版本更清楚。

### 2.1 直觉解释

| 指标 | 含义 |
| --- | --- |
| IC 均值 | 因子平均预测方向和强度 |
| IC 标准差 | 因子预测能力的波动 |
| ICIR | 因子预测能力的稳定性 |

一个因子：

- 平均 IC 高
- IC 波动低
- 正 IC 占比高
- 滚动 IC 不长期翻负

这样的因子才更值得深入研究。

---

## 3. 准备 Python 环境


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260705)


如果缺包，可以先安装：


In [ ]:
pip install numpy pandas matplotlib


---

## 4. 构造三种不同稳定性的因子

为了理解 ICIR，我们模拟三类因子：

| 因子 | 特征 |
| --- | --- |
| steady_factor | 平均 IC 不夸张，但比较稳定 |
| noisy_factor | 平均有点正，但波动很大 |
| decaying_factor | 前期有效，后期逐渐失效 |

### 4.1 构造样本数据


In [ ]:
dates = pd.bdate_range("2024-01-02", periods=220)
tickers = [f"Stock_{i:03d}" for i in range(400)]

records = []

for t, date in enumerate(dates):
    n = len(tickers)

    steady_factor = rng.normal(0, 1, size=n)
    noisy_factor = rng.normal(0, 1, size=n)
    decaying_factor = rng.normal(0, 1, size=n)

    # 稳定因子：每天都有类似强度
    steady_strength = 0.018

    # 噪声因子：每天强度忽高忽低，甚至会反向
    noisy_strength = rng.normal(0.012, 0.030)

    # 衰减因子：前期强，后期弱
    decaying_strength = max(0.025 * (1 - t / len(dates)), 0)

    noise = rng.normal(0, 0.065, size=n)

    future_20d_ret = (
        steady_strength * steady_factor
        + noisy_strength * noisy_factor
        + decaying_strength * decaying_factor
        + noise
    )

    for i, ticker in enumerate(tickers):
        records.append((
            date,
            ticker,
            steady_factor[i],
            noisy_factor[i],
            decaying_factor[i],
            future_20d_ret[i],
        ))

data = pd.DataFrame(
    records,
    columns=[
        "date",
        "ticker",
        "steady_factor",
        "noisy_factor",
        "decaying_factor",
        "future_20d_ret",
    ]
)

data.head()


---

## 5. 计算每日 IC

先复用第 3 天的函数。


In [ ]:
def calc_daily_ic(
    data: pd.DataFrame,
    factor_col: str,
    return_col: str = "future_20d_ret",
    method: str = "spearman",
    min_count: int = 30,
) -> pd.Series:
    ic_values = {}

    for date, group in data.groupby("date"):
        g = group[[factor_col, return_col]].dropna()

        if len(g) < min_count:
            ic_values[date] = np.nan
            continue

        ic_values[date] = g[factor_col].corr(g[return_col], method=method)

    return pd.Series(ic_values, name=factor_col).sort_index()


计算三条 Rank IC 序列：


In [ ]:
ic_df = pd.DataFrame({
    "steady_factor": calc_daily_ic(data, "steady_factor"),
    "noisy_factor": calc_daily_ic(data, "noisy_factor"),
    "decaying_factor": calc_daily_ic(data, "decaying_factor"),
})

ic_df.head()


---

## 6. ICIR 计算函数


In [ ]:
def calc_icir(ic_series: pd.Series) -> float:
    clean = ic_series.dropna()
    if clean.std() == 0:
        return np.nan
    return clean.mean() / clean.std()


def ic_stability_report(ic_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for col in ic_df.columns:
        s = ic_df[col].dropna()
        rows.append({
            "factor": col,
            "mean_ic": s.mean(),
            "std_ic": s.std(),
            "icir": calc_icir(s),
            "positive_ratio": (s > 0).mean(),
            "min_ic": s.min(),
            "max_ic": s.max(),
            "count": s.count(),
        })

    return pd.DataFrame(rows).set_index("factor")


report = ic_stability_report(ic_df)
report


解释：

- `mean_ic` 高：平均预测关系强。
- `std_ic` 低：波动小。
- `icir` 高：稳定性更好。
- `positive_ratio` 高：方向更一致。

---

## 7. 画 IC 时间序列


In [ ]:
ic_df.plot(title="Daily Rank IC")
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Rank IC")
plt.show()


观察：

- `steady_factor` 应该更像围绕正值上下波动。
- `noisy_factor` 可能大起大落。
- `decaying_factor` 可能前期较好，后期变弱。

---

## 8. 滚动 IC：看阶段性变化

平均值会掩盖阶段变化。

比如一个因子前半年有效、后半年失效，全年均值可能看起来还行。  
滚动 IC 可以帮你看到这种变化。

### 8.1 计算滚动均值


In [ ]:
rolling_window = 20
rolling_ic_mean = ic_df.rolling(rolling_window).mean()

rolling_ic_mean.tail()


### 8.2 画滚动 IC


In [ ]:
rolling_ic_mean.plot(title=f"{rolling_window}-day Rolling Mean Rank IC")
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Rolling mean IC")
plt.show()


如果滚动 IC 长期掉到 0 以下，说明因子可能阶段性失效。

### 8.3 计算滚动 ICIR


In [ ]:
rolling_icir = ic_df.rolling(rolling_window).mean() / ic_df.rolling(rolling_window).std()
rolling_icir.tail()


In [ ]:
rolling_icir.plot(title=f"{rolling_window}-day Rolling ICIR")
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Rolling ICIR")
plt.show()


滚动 ICIR 比滚动均值更严格，因为它同时惩罚波动。

---

## 9. 累计 IC：看长期方向

累计 IC 不是收益曲线，但它很适合看因子预测能力是否在长期累积。


In [ ]:
cumulative_ic = ic_df.fillna(0).cumsum()

cumulative_ic.plot(title="Cumulative Rank IC")
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Cumulative IC")
plt.show()


如果累计 IC 曲线持续向上，说明因子长期倾向于正向预测。  
如果曲线反复横盘或下行，就要小心。

---

## 10. IC 分布图

看时间序列之外，也可以看 IC 分布。


In [ ]:
for col in ic_df.columns:
    ic_df[col].hist(alpha=0.45, bins=30, label=col)

plt.axvline(0, color="black", linewidth=1)
plt.title("Distribution of Daily Rank IC")
plt.xlabel("Rank IC")
plt.ylabel("Frequency")
plt.legend()
plt.show()


一个更稳的因子，IC 分布通常会：

- 均值在 0 右侧
- 大部分面积在 0 右侧
- 左尾不要太夸张

---

## 11. 用分年度视角看稳定性

虽然我们的模拟数据只有 220 个交易日，但仍然可以按月份观察。


In [ ]:
monthly_ic = ic_df.resample("ME").mean()
monthly_ic


In [ ]:
monthly_ic.plot(kind="bar", title="Monthly Average Rank IC")
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Monthly mean IC")
plt.xticks(rotation=45)
plt.show()


真实研究里建议进一步看：

- 分年度 IC
- 牛市、熊市、震荡市 IC
- 大盘、小盘股票池 IC
- 不同行业 IC
- 不同持有期 IC

---

## 12. 平均 IC 与 ICIR 的冲突

有时你会看到：


因子 A：平均 IC 更高，但 ICIR 更低
因子 B：平均 IC 较低，但 ICIR 更高


怎么选？

没有绝对答案，但可以这样理解：

- 如果你追求长期稳定，ICIR 更重要。
- 如果你能识别因子适用环境，平均 IC 高但不稳定的因子也可能有价值。
- 如果一个因子平均 IC 高得离谱，要先查未来函数和样本偏差。

因子研究不是单指标打分。

更好的判断方式是：


IC 均值 + ICIR + 正 IC 占比 + 滚动 IC + 分组回测 + 交易成本


---

## 13. 今日目标产出：ICIR 分析图

下面写一个函数，一次生成常见 ICIR 分析图。


In [ ]:
def plot_icir_dashboard(ic_df: pd.DataFrame, rolling_window: int = 20) -> pd.DataFrame:
    report = ic_stability_report(ic_df)
    rolling_mean = ic_df.rolling(rolling_window).mean()
    cumulative = ic_df.fillna(0).cumsum()

    fig, axes = plt.subplots(3, 1, figsize=(11, 12), sharex=True)

    ic_df.plot(ax=axes[0], title="Daily Rank IC")
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set_ylabel("IC")

    rolling_mean.plot(ax=axes[1], title=f"{rolling_window}-day Rolling Mean IC")
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_ylabel("Rolling IC")

    cumulative.plot(ax=axes[2], title="Cumulative IC")
    axes[2].axhline(0, color="black", linewidth=1)
    axes[2].set_ylabel("Cumulative IC")

    plt.tight_layout()
    plt.show()

    return report


dashboard_report = plot_icir_dashboard(ic_df, rolling_window=20)
dashboard_report


这就是今天的目标产出：  
一张能同时看每日 IC、滚动 IC、累计 IC 的分析图，再配一张 ICIR 汇总表。

---

## 14. 如何判断一个 ICIR 是否“好”？

没有跨市场、跨频率、跨股票池的绝对标准。

粗略经验：

| 非年化 ICIR | 直觉 |
| ---: | --- |
| < 0 | 方向可能反了 |
| 0 - 0.2 | 很弱，稳定性不足 |
| 0.2 - 0.5 | 有观察价值 |
| 0.5 - 1.0 | 比较稳定，值得深入 |
| > 1.0 | 很强，需要检查是否过拟合或未来函数 |

注意：

> ICIR 不是越高越无脑好。异常好看的结果通常先查错。

---

## 15. IC 稳定性检查清单

### 15.1 看均值

平均 IC 是否稳定为正？

### 15.2 看标准差

IC 是否大起大落？

### 15.3 看正 IC 占比

正 IC 日期占比是否明显超过 50%？

### 15.4 看滚动 IC

是否有长时间低于 0？

### 15.5 看累计 IC

累计 IC 是否长期向上？

### 15.6 看分阶段

是否只在某个阶段有效？

### 15.7 看不同股票池

大盘、小盘、全市场是否表现一致？

### 15.8 看不同持有期

5 日、20 日、60 日 IC 是否衰减合理？

---

## 16. 今天的知识图谱


In [ ]:
mindmap
  root((ICIR))
    输入
      每日IC序列
      RankIC
      PearsonIC
    公式
      IC均值
      IC标准差
      mean除以std
    稳定性
      正IC占比
      滚动IC
      滚动ICIR
      累计IC
    图表
      每日IC曲线
      滚动均值曲线
      累计IC曲线
      IC分布图
      月度IC图
    解读
      高均值低稳定
      低均值高稳定
      因子衰减
      阶段失效
    风险点
      样本太短
      未来函数
      单一市场阶段
      过拟合


---

## 17. 初学者最容易踩的 8 个坑

### 坑 1：只看平均 IC

平均 IC 可能由少数极端日期贡献。稳定性同样重要。

### 坑 2：把 ICIR 当作收益指标

ICIR 衡量预测稳定性，不是策略收益率。

### 坑 3：样本太短就下结论

几十天 IC 很容易受偶然影响。

### 坑 4：看到滚动 IC 下行还不追问原因

这可能是因子拥挤、市场风格切换、数据问题或逻辑失效。

### 坑 5：忽略正 IC 占比

均值为正但正 IC 占比不高，说明稳定性可能一般。

### 坑 6：把年化和非年化 ICIR 混着比较

比较前先确认计算口径。

### 坑 7：不做分阶段分析

一个因子可能只在小盘股、熊市或流动性宽松阶段有效。

### 坑 8：ICIR 很漂亮就直接相信

异常漂亮的结果，优先排查未来函数、标签错位和样本污染。

---

## 18. 今天的动手作业

### 作业 A：解释 ICIR

用自己的话解释：

1. ICIR 为什么要除以 IC 标准差？
2. ICIR 和平均 IC 的区别是什么？
3. ICIR 高但平均 IC 低，可能说明什么？

### 作业 B：运行代码

运行本文所有 Python 代码，记录三个因子的：

- mean IC
- std IC
- ICIR
- positive ratio

### 作业 C：调整噪声

把 `future_20d_ret` 里的噪声标准差从 `0.065` 改成：


In [ ]:
0.03
0.10


观察 ICIR 如何变化。

### 作业 D：改变滚动窗口

把滚动窗口改成：


In [ ]:
10
40
60


观察滚动 IC 曲线变得更敏感还是更平滑。

### 作业 E：写自己的 ICIR 报告函数

要求输出：


factor, mean_ic, std_ic, icir, positive_ratio, count


---

## 19. 自测题

### 题 1

ICIR 的公式是什么？

答案：`IC 均值 / IC 标准差`。

### 题 2

平均 IC 高但 ICIR 低，说明什么？

答案：说明预测能力可能不稳定，IC 波动较大。

### 题 3

滚动 IC 有什么用？

答案：观察因子在不同阶段的有效性变化，发现衰减或阶段性失效。

### 题 4

累计 IC 是策略净值吗？

答案：不是。累计 IC 是预测能力的累积，不是可交易收益曲线。

### 题 5

ICIR 很高是否一定说明因子能赚钱？

答案：不一定。还需要分层回测、组合构建、交易成本和风险控制验证。

---

## 20. 今日复盘模板


第 04 天复盘：ICIR

1. 我今天理解的 ICIR：

2. 我计算出的 mean IC：

3. 我计算出的 ICIR：

4. 滚动 IC 图给我的信息：

5. 累计 IC 图给我的信息：

6. 我认为最稳定的因子：

7. 明天学习分层回测前，我需要准备：


---

## 21. 明天预告：分层回测

IC 和 ICIR 告诉我们因子是否有预测关系。  
但投资最终要看组合表现。

明天会问：

> 如果按因子值把股票分成 5 组或 10 组，高分组是否真的跑赢低分组？

这就是分层回测。

---

## 22. 一句话收尾

平均 IC 像一个人的平均成绩，ICIR 像这个人成绩的稳定性。

> 因子研究里，偶尔聪明不够，稳定地有一点聪明更珍贵。

---

## 23. 仅供学习的提醒

本文所有示例使用模拟数据，仅用于解释 ICIR 和 IC 稳定性分析，不构成任何投资建议。真实研究还需要处理数据质量、股票池变化、行业市值中性化、交易成本和样本外检验。

---

# 统一高质量增强模块

> 本增强模块用于把第 04 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：ICIR
- 必做：IC稳定性分析
- 选做：滚动IC
- 目标产出：ICIR分析图

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

一个因子偶尔 IC 很高，但大多数时候乱跳；另一个因子 IC 不夸张但长期为正。长期策略更喜欢后者。

这个例子背后的关键直觉是：

> 平均有效不够，稳定有效才珍贵。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


ICIR
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── ICIR分析图


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(104)
ic = pd.Series(rng.normal(0.035, 0.08, 180), index=pd.bdate_range("2024-01-02", periods=180))
rolling_mean = ic.rolling(20).mean()
rolling_icir = ic.rolling(20).mean() / ic.rolling(20).std()
report = pd.Series({
    "mean_ic": ic.mean(),
    "std_ic": ic.std(),
    "icir": ic.mean() / ic.std(),
    "positive_ratio": (ic > 0).mean(),
    "latest_rolling_ic": rolling_mean.dropna().iloc[-1],
    "latest_rolling_icir": rolling_icir.dropna().iloc[-1],
})
print(report.round(4))


## E. 产出验收标准

完成今天课程后，你的 `ICIR分析图` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `ICIR` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `ICIR分析图` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `ICIR分析图`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 04 天复盘：ICIR

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
